In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from  sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score

In [2]:
df=pd.read_csv('../data/student_placement.csv')
func_trans=FunctionTransformer(func=np.log1p)

In [3]:
df['technical_scr']=0.35*df["coding_skills"]+0.35*df['dsa_score']+0.15*df['ml_knowledge']+0.15*df['system_design']
df['experience_scr']=df["internships"]*3+df['projects_count']*2+df['hackathons']+df['open_source_contributions']/10
df['academic_scr']=df['cgpa']*10-df['backlogs']*15
df['dsa_coding']=(df['coding_skills']*df['dsa_score'])/100
df['resume_strgth']=df["projects_count"]+df["internships"]*2+df["hackathons"]+df["open_source_contributions"]/10
df['salary_package_lpa']=func_trans.fit_transform(df["salary_package_lpa"])

In [4]:
# total_students=len(df)
# placed=len(df.placement_status[df.placement_status==1])
# plc_rate=placed/total_students*100
# avg_pkg=df[df.placement_status!=0]['salary_package_lpa'].mean()
# max_pkg=df.salary_package_lpa.max()
# median_pkg=df.salary_package_lpa.median()

In [5]:
# sns.set_style(style='whitegrid')

In [6]:
# plt.pie(df['placement_status'].value_counts(),labels=['Placed','Not Placed'],autopct='%1.1f%%')

In [7]:
# sns.histplot(df['cgpa'])

In [8]:
# placed_rate=df.groupby('branch')['placement_status'].mean()*100
# print(placed_rate)

In [9]:
x_train1,x_test1,y_train1,y_test1=train_test_split(df.drop(columns=['placement_status','salary_package_lpa']),df['placement_status'],test_size=0.2,random_state=42,stratify=df['placement_status'])

In [10]:
ohe_branch=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
lbl_tier=OrdinalEncoder(categories=[['Tier 3','Tier 2','Tier 1']])
x_train1_branch=ohe_branch.fit_transform(x_train1[['branch']])
x_train1_tier=lbl_tier.fit_transform(x_train1[['college_tier']])
x_test1_branch=ohe_branch.transform(x_test1[['branch']])
x_test1_tier=lbl_tier.transform(x_test1[['college_tier']])

In [11]:
x_train1_rem=x_train1.drop(columns=['college_tier','branch'])
x_test1_rem=x_test1.drop(columns=['college_tier','branch'])

In [12]:
x_train1_trans=np.concatenate((x_train1_rem,x_train1_branch,x_train1_tier),axis=1)
x_test1_trans=np.concatenate((x_test1_rem,x_test1_branch,x_test1_tier),axis=1)

In [13]:
mdl1=XGBClassifier(n_estimators=500,learning_rate=0.05,min_child_weight=3,max_depth=5,subsample=0.8,colsample_bytree=0.8,gamma=0.2,random_state=42)
mdl1.fit(x_train1_trans,y_train1)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [14]:
y_pred=mdl1.predict(x_test1_trans)

In [15]:
accuracy_score(y_test1,y_pred)

0.8863076923076924

In [16]:
train_pred = mdl1.predict(x_train1_trans)
print("Train Accuracy:", accuracy_score(y_train1, train_pred))
print("Test Accuracy:", accuracy_score(y_test1, y_pred))

Train Accuracy: 0.903201923076923
Test Accuracy: 0.8863076923076924


In [17]:
# scores=cross_val_score(mdl1,x_train1_trans,y_train1,cv=5)
# print(scores.mean())

In [18]:
df.columns

Index(['cgpa', 'backlogs', 'coding_skills', 'dsa_score', 'aptitude_score',
       'communication_skills', 'ml_knowledge', 'system_design', 'internships',
       'projects_count', 'hackathons', 'open_source_contributions', 'branch',
       'college_tier', 'placement_status', 'salary_package_lpa',
       'technical_scr', 'experience_scr', 'academic_scr', 'dsa_coding',
       'resume_strgth'],
      dtype='str')

In [19]:
df

,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,...,open_source_contributions,branch,college_tier,placement_status,salary_package_lpa,technical_scr,experience_scr,academic_scr,dsa_coding,resume_strgth
0,6.87,0,64.8,61.2,59.2,45.6,35.5,50.0,0,2,...,0,Mechanical,Tier 2,1,1.656321,56.925,6.0,68.7,39.6576,4.0
1,5.90,0,68.9,62.2,51.2,55.7,73.6,60.9,1,6,...,3,AIML,Tier 1,1,2.613740,66.060,17.3,59.0,42.8558,10.3
2,7.33,0,74.0,77.2,76.6,72.5,56.3,77.6,1,7,...,1,Civil,Tier 1,1,2.630449,73.005,19.1,73.3,57.1280,11.1
3,8.43,0,75.2,74.5,72.6,49.0,64.4,73.6,0,4,...,0,Mechanical,Tier 2,0,0.000000,73.095,8.0,84.3,56.0240,4.0
4,7.07,0,70.1,63.5,56.0,47.0,34.7,64.6,3,5,...,3,CSE,Tier 3,1,2.386926,61.655,20.3,70.7,44.5135,12.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129995,8.82,0,71.7,63.5,81.1,66.7,55.1,83.7,3,5,...,3,Mechanical,Tier 2,1,2.873000,68.140,22.3,88.2,45.5295,14.3
129996,8.75,0,80.2,81.1,74.8,61.5,67.4,66.0,1,3,...,0,IT,Tier 2,1,2.599722,76.465,9.0,87.5,65.0422,5.0
129997,6.99,0,70.4,57.1,61.2,65.9,52.8,71.9,4,5,...,1,CSE,Tier 2,1,2.603430,63.330,22.1,69.9,40.1984,13.1
129998,8.23,0,92.5,87.0,42.3,33.5,78.5,57.7,2,6,...,1,CSE,Tier 2,1,3.054001,83.255,20.1,82.3,80.4750,12.1


In [20]:
x_train2,x_test2,y_train2,y_test2=train_test_split(df[df.placement_status==1].drop(columns=['placement_status','salary_package_lpa']),df[df.placement_status==1][['salary_package_lpa']],test_size=0.2,random_state=42)

In [21]:
x_test2.columns

Index(['cgpa', 'backlogs', 'coding_skills', 'dsa_score', 'aptitude_score',
       'communication_skills', 'ml_knowledge', 'system_design', 'internships',
       'projects_count', 'hackathons', 'open_source_contributions', 'branch',
       'college_tier', 'technical_scr', 'experience_scr', 'academic_scr',
       'dsa_coding', 'resume_strgth'],
      dtype='str')

In [22]:
x_train2_branch=ohe_branch.fit_transform(x_train2[['branch']])
x_train2_college=lbl_tier.fit_transform(x_train2[['college_tier']])
x_test2_branch=ohe_branch.transform(x_test2[['branch']])
x_test2_college=lbl_tier.transform(x_test2[['college_tier']])

In [23]:
x_train2_rem=x_train2.drop(columns=['college_tier','branch'])
x_test2_rem=x_test2.drop(columns=['college_tier','branch'])

In [24]:
x_train2_trans=np.concatenate((x_train2_rem,x_train2_branch,x_train2_college),axis=1)
x_test2_trans=np.concatenate((x_test2_rem,x_test2_branch,x_test2_college),axis=1)

In [25]:
mdl2=XGBRegressor(n_estimators=500,learning_rate=0.05,min_child_weight=3,max_depth=5,subsample=0.8,colsample_bytree=0.8,gamma=0.2,random_state=42,n_jobs=-1,eval_metric='rmse')
mdl2.fit(x_train2_trans,y_train2)


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [26]:
y_pred2=mdl2.predict(x_test2_trans)
y_pred_lpa = np.expm1(y_pred2)
y_test_lpa = np.expm1(y_test2.values.ravel())

In [27]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

print("R² :", r2_score(y_test_lpa, y_pred_lpa))
print("MAE:", mean_absolute_error(y_test_lpa, y_pred_lpa))
print("RMSE:", root_mean_squared_error(y_test_lpa, y_pred_lpa))

R² : 0.9321361804838898
MAE: 0.7794565028347115
RMSE: 1.9550691784655856
